In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class CachingAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_head

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, paste_kv_cache=None):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # 2. KV Cache Logic
        if paste_kv_cache is not None:
            past_k, past_v = paste_kv_cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        current_kv_cache = (k, v)
        total_seq_len = k.shape[2]
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if T > 1:
            mask = torch.tril(torch.once(T, T, device=x.device)).view(1, 1, T, T)
            scores = scores.masked_fill(mask == 0, float('-int'))
        attn_weight = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, v)

        context = context.transpose(1, 2).contiguous().view(B, T, C)
        output = self.out_proj(context)
        
        return output, current_kv_cache

# Exercise

In [8]:
import torch
import torch.nn as nn
import math

class CachingAttention(nn.Module):
    def __init__(self, d_model=256, n_heads=4):
        super().__init__()
        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, S, D = x.shape
        x = x.view(B, S, self.n_heads, self.head_dim)
        return x.transpose(1, 2)

    def combine_heads(self, x):
        B, H, S, Hd = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(B, S, H * Hd)

    def forward(self, x, past_cache=None):
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = self.split_heads(q)
        k = self.split_heads(k)
        v = self.split_heads(v)

        if past_cache is not None:
            past_k, past_v = past_cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v)
        
        scores = torch.matmul(q, k.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        out = self.combine_heads(out)
        out = self.out_proj(out)

        return out, new_cache

In [9]:
past_k = torch.randn(1, 4, 5, 64)
past_v = torch.randn(1, 4, 5, 64)

x_new = torch.randn(1, 1, 256)
attn = CachingAttention(d_model=256, n_heads=4)

output, new_cache = attn(x_new, (past_k, past_v))
new_k, new_v = new_cache

In [10]:
# =========================================================
# Print shapes
# =========================================================

print("Output shape:", output.shape)
print("Updated key cache shape:", new_k.shape)
print("Updated value cache shape:", new_v.shape)

# =========================================================
# Assertions
# =========================================================

assert output.shape == (1, 1, 256)

# Old seq_len = 5
# New token adds +1
assert new_k.shape == (1, 4, 6, 64)
assert new_v.shape == (1, 4, 6, 64)

print("\nAll shape checks passed.")

Output shape: torch.Size([1, 1, 256])
Updated key cache shape: torch.Size([1, 4, 6, 64])
Updated value cache shape: torch.Size([1, 4, 6, 64])

All shape checks passed.
